# Lista 7

## scikit i Optuna

(6pkt + 3pkt)

Na liście znajduje się 5 zadań. Zadanie *optuna* jest za 2pkt, a inne zadania za 1pkt. Po rozwiązaniu zadania pokaż kod prowadzącemu i odpowiedz na **pytanie kontrolne** — tylko wtedy przyznajemy punkty. Dodatkowo prześlij zadanie na platformie skos.

Zadania domowe oznaczone są 🏠. Za wykonanie każdego z nich otrzymasz 1pkt. 

## KFold

Wykonaj walidację krzyżową `KFold` (5 foldów) dla modelu `LinearRegression` na zbiorze California Housing.
Zapisz MSE dla każdego folda i podaj średnie MSE.

In [1]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np

X, y = fetch_california_housing(return_X_y=True)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
model = LinearRegression()

mse_scores = []

for train_idx, test_idx in kf.split(X):
    model.fit(X[train_idx], y[train_idx])
    preds = model.predict(X[test_idx])
    mse_scores.append(mean_squared_error(y[test_idx], preds))

print("MSE fold-by-fold:", mse_scores)
print("Średnie MSE:", np.mean(mse_scores))

MSE fold-by-fold: [0.5558915986952405, 0.5276562547736293, 0.5092832097248735, 0.5048507784142153, 0.5551804780114787]
Średnie MSE: 0.5305724639238876


## StratifiedKFold

Na zbiorze Iris zbuduj problem binarny: „Setosa” vs „inne”.
Porównaj proporcje klas w foldach między `KFold` a `StratifiedKFold`.
Przeprowadź walidację `LogisticRegression` (accuracy).

In [2]:
from sklearn.datasets import load_iris
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression

X, y_raw = load_iris(return_X_y=True)
y = (y_raw == 0).astype(int)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Udział klasy pozytywnej (KFold):")
for _, test_idx in kf.split(X):
    print(y[test_idx].mean())

print("\nUdział klasy pozytywnej (StratifiedKFold):")
for _, test_idx in skf.split(X, y):
    print(y[test_idx].mean())

model = LogisticRegression(max_iter=200)
scores = cross_val_score(model, X, y, cv=skf)
print("\nAccuracy:", scores.mean())

Udział klasy pozytywnej (KFold):
0.3333333333333333
0.43333333333333335
0.4
0.26666666666666666
0.23333333333333334

Udział klasy pozytywnej (StratifiedKFold):
0.3333333333333333
0.3333333333333333
0.3333333333333333
0.3333333333333333
0.3333333333333333

Accuracy: 1.0


## GridSearchCV

Wykonaj GridSearchCV dla `SVC`.
Siatka parametrów:

* C: [0.1, 1, 10]
* gamma: ["scale", "auto"]
* kernel: ["rbf", "poly"]

Użyj `StratifiedKFold(5)`. Wyświetl najlepsze parametry i wynik.

In [3]:
from sklearn.datasets import load_iris
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.svm import SVC

X, y = load_iris(return_X_y=True)

param_grid = {
    'C': [0.1, 1, 10],
    'gamma': ['scale', 'auto'],
    'kernel': ['linear', 'rbf']
}

grid = GridSearchCV(SVC(), param_grid, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42))
grid.fit(X, y)

print("Najlepsze parametry:", grid.best_params_)
print("Najlepszy wynik:", grid.best_score_)

Najlepsze parametry: {'C': 1, 'gamma': 'scale', 'kernel': 'linear'}
Najlepszy wynik: 0.9866666666666667


## RandomizedSearchCV

Wykonaj RandomizedSearchCV dla `RandomForestClassifier`.
Parametry losowane:

* n_estimators: 100–999
* max_depth: 2–39

30 losowań. Wypisz najlepsze parametry i accuracy.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from scipy.stats import randint

X, y = load_iris(return_X_y=True)

param_dist = {
    'n_estimators': randint(100, 1000),
    'max_depth': randint(2, 40)
}

rnd = RandomizedSearchCV(RandomForestClassifier(), param_dist, n_iter=30, random_state=42)

rnd.fit(X, y)

print("Najlepsze parametry:", rnd.best_params_)
print("Najlepszy wynik:", rnd.best_score_)

Najlepsze parametry: {'max_depth': 30, 'n_estimators': 370}
Najlepszy wynik: 0.9666666666666668


## Optuna

Twoim zadaniem jest przeprowadzić pełną optymalizację hiperparametrów modelu `GradientBoostingClassifier` z użyciem Optuny.

Wykonaj:

1. Wczytaj dane z `sklearn.datasets`.
2. Zdefiniuj funkcję celu, w której:

   * pobierzesz hiperparametry z `trial.suggest_*`,
   * stworzysz model `GradientBoostingClassifier`,
   * ocenisz go 5-krotną walidacją krzyżową,
   * zwrócisz średnią dokładność.
3. Uruchom optymalizację z `TPESampler` + `MedianPruner`, ustawiając parametry:

   * `direction="maximize"`
   * `storage="sqlite:///gb_optuna.db"`
   * `load_if_exists=True`
4. Uruchom drugą optymalizację z `RandomSampler` (tylko porównanie, bez zapisu do bazy).
5. Wygeneruj kilka (minimum 3) wykresy z `optuna.visualization` na podstawie study zapisanej w SQLite.
6. Uruchom Optuna Dashboard na bazie `gb_optuna.db`.
7. Wydrukuj porównanie wyników TPE vs RandomSampler.

In [4]:
import optuna
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.datasets import load_breast_cancer

# ---------------------------------------------------------------
# 1) Wczytanie danych
# ---------------------------------------------------------------

X, y = load_breast_cancer(return_X_y=True)

# ---------------------------------------------------------------
# 2) Funkcja celu używana we wszystkich optymalizacjach
# ---------------------------------------------------------------

def objective(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 400),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "max_depth": trial.suggest_int("max_depth", 1, 6),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20)
    }

    model = GradientBoostingClassifier(**params)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")

    return scores.mean()

# ---------------------------------------------------------------
# 3) Główna optymalizacja — TPE + zapis do SQLite
# ---------------------------------------------------------------

study_sqlite = optuna.create_study(
    study_name="gb_sqlite_study",
    storage="sqlite:///gb_optuna.db",
    direction="maximize",
    load_if_exists=True,
    sampler= optuna.samplers.TPESampler(seed=42),
    pruner= optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=0, interval_steps=1),
    
)

study_sqlite.optimize(objective, n_trials=20)

print("Najlepsze parametry TPE:", study_sqlite.best_params)
print("Najlepsze accuracy TPE:", study_sqlite.best_value)

# ---------------------------------------------------------------
# 4) Druga optymalizacja — RandomSampler (bez zapisu do bazy)
# ---------------------------------------------------------------

study_random = optuna.create_study(
    study_name="gb_random_study",
    storage=None,
    direction="maximize",
    load_if_exists=True,
    sampler=optuna.samplers.RandomSampler(seed=42),
)

study_random.optimize(objective, n_trials=20)

print("Najlepsze accuracy RandomSampler:", study_random.best_value)

[I 2026-11-16 15:41:41,150] A new study created in RDB with name: gb_sqlite_study
[I 2026-11-16 15:41:42,849] Trial 0 finished with value: 0.9578481602235678 and parameters: {'n_estimators': 181, 'learning_rate': 0.28570714885887566, 'max_depth': 5, 'subsample': 0.7993292420985183, 'min_samples_split': 4}. Best is trial 0 with value: 0.9578481602235678.
[I 2026-11-16 15:41:44,345] Trial 1 finished with value: 0.9508150908244062 and parameters: {'n_estimators': 104, 'learning_rate': 0.026844247528777843, 'max_depth': 6, 'subsample': 0.8005575058716043, 'min_samples_split': 15}. Best is trial 0 with value: 0.9578481602235678.
[I 2026-11-16 15:41:44,889] Trial 2 finished with value: 0.9631268436578171 and parameters: {'n_estimators': 57, 'learning_rate': 0.29127385712697834, 'max_depth': 5, 'subsample': 0.6061695553391381, 'min_samples_split': 5}. Best is trial 2 with value: 0.9631268436578171.
[I 2026-11-16 15:41:45,919] Trial 3 finished with value: 0.9595870206489675 and parameters: {'n

Najlepsze parametry TPE: {'n_estimators': 343, 'learning_rate': 0.2593247050447376, 'max_depth': 3, 'subsample': 0.5382147727693088, 'min_samples_split': 11}
Najlepsze accuracy TPE: 0.9718987734823784


[I 2026-11-16 15:42:16,273] Trial 0 finished with value: 0.9596025461884802 and parameters: {'n_estimators': 181, 'learning_rate': 0.28570714885887566, 'max_depth': 5, 'subsample': 0.7993292420985183, 'min_samples_split': 4}. Best is trial 0 with value: 0.9596025461884802.
[I 2026-11-16 15:42:17,692] Trial 1 finished with value: 0.9578326346840551 and parameters: {'n_estimators': 104, 'learning_rate': 0.026844247528777843, 'max_depth': 6, 'subsample': 0.8005575058716043, 'min_samples_split': 15}. Best is trial 0 with value: 0.9596025461884802.
[I 2026-11-16 15:42:18,243] Trial 2 finished with value: 0.9578481602235678 and parameters: {'n_estimators': 57, 'learning_rate': 0.29127385712697834, 'max_depth': 5, 'subsample': 0.6061695553391381, 'min_samples_split': 5}. Best is trial 0 with value: 0.9596025461884802.
[I 2026-11-16 15:42:19,272] Trial 3 finished with value: 0.9648657040832169 and parameters: {'n_estimators': 114, 'learning_rate': 0.09823025045826593, 'max_depth': 4, 'subsampl

Najlepsze accuracy RandomSampler: 0.9648657040832169


In [5]:
# ---------------------------------------------------------------
# 5) Wizualizacje
# ---------------------------------------------------------------

from optuna.visualization import (
    plot_optimization_history,
    plot_parallel_coordinate,
    plot_slice,
    plot_contour,
    plot_param_importances,
    plot_intermediate_values,
    plot_edf,
    plot_timeline,
    plot_rank,
)

plot_optimization_history(study_sqlite).show()
plot_parallel_coordinate(study_sqlite).show()
plot_slice(study_sqlite).show()

# ---------------------------------------------------------------
# 6) Dashboard — komenda do uruchomienia w terminalu:
# ---------------------------------------------------------------
# optuna-dashboard sqlite:///gb_optuna.db

# ---------------------------------------------------------------
# 7) Porównanie wyników
# ---------------------------------------------------------------

print("===============================================")
print("PORÓWNANIE")
print("TPE (SQLite):   ", study_sqlite.best_value)
print("RandomSampler:   ", study_random.best_value)
print("===============================================")

PORÓWNANIE
TPE (SQLite):    0.9718987734823784
RandomSampler:    0.9648657040832169


## 🏠 GroupKFold

Wygeneruj zbiór danych składający się z **500 próbek** oraz **10 cech**.
Następnie przydziel każdą próbkę do jednej z **50 grup**, tak aby każda grupa zawierała dokładnie **10 próbek**.

Celem zadania jest sprawdzenie, jak działa walidacja krzyżowa z podziałem na grupy.
Użyj **GroupKFold(5)** – oznacza to, że dane będą dzielone tak, aby **żadne próbki z tej samej grupy nie trafiły jednocześnie do zbioru treningowego i testowego**.

Na tak przygotowanych danych wykonaj walidację krzyżową modelu **RandomForestClassifier**, a następnie wypisz średnią dokładność.

In [1]:
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, cross_val_score
import numpy as np

# 1) Generowanie sztucznych danych
X, y = make_classification(n_samples=500, n_features=10, random_state=42)

# 2) Przydział 500 próbek do 50 grup (po 10 próbek w każdej)
groups = np.repeat(np.arange(50), 10)

gkf = GroupKFold(n_splits=5)
model = RandomForestClassifier(random_state=42)

scores = cross_val_score(model, X, y, cv=gkf, groups=groups)
print("Accuracy:", scores.mean())

Accuracy: 0.9460000000000001


## 🏠 TimeSeriesSplit

Wygeneruj **sztuczny szereg czasowy** składający się z 1000 obserwacji.
Przyjmij, że cechą wejściową jest indeks czasu (0,1,2,…,999), a wartością docelową sygnał sinusoidalny z zakłóceniem (szumem gaussowskim).

Następnie zastosuj **TimeSeriesSplit(5)**, który wykonuje walidację krzyżową dostosowaną do danych sekwencyjnych — każdy kolejny fold używa coraz większej części historii jako zbioru treningowego.

Na każdym foldzie wytrenuj model **GradientBoostingRegressor**, następnie policz i wypisz **MAE (Mean Absolute Error)** na zbiorze testowym.

Wynik powinien zawierać wartość MAE dla każdego folda osobno.

In [2]:
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

# 1) Generowanie sztucznego szeregu czasowego
np.random.seed(42)
X = np.arange(1000).reshape(-1, 1)
y = np.sin(X[:, 0] / 20) + np.random.normal(scale=0.1, size=1000)

tscv = TimeSeriesSplit(n_splits=5)
model = GradientBoostingRegressor()

for i, (train_idx, test_idx) in enumerate(tscv.split(X)):
    model.fit(X[train_idx], y[train_idx])
    preds = model.predict(X[test_idx])
    print(f"Fold {i}: MAE = {mean_absolute_error(y[test_idx], preds)}")

Fold 0: MAE = 0.8288663064508833
Fold 1: MAE = 0.7330599469306064
Fold 2: MAE = 0.6298690477081538
Fold 3: MAE = 0.9575344251525488
Fold 4: MAE = 0.6518714737382351


## 🏠 Optymalizacja hiperparametrów SVM z użyciem Optuny

Twoim zadaniem jest zoptymalizować hiperparametry modelu **Support Vector Machine (SVC)** z wykorzystaniem biblioteki **Optuna**.

Wykonaj następujące kroki:

1. Wczytaj zbiór danych **Wine** ze `sklearn.datasets`.
2. Podziel dane na zbiór treningowy i testowy (np. 80/20).
3. Zdefiniuj funkcję celu `objective(trial)`, która:

   * wybierze hiperparametry `C` oraz `gamma` (oba w skali logarytmicznej),
   * wybierze kernel spośród: `"rbf"`, `"poly"`, `"sigmoid"`,
   * utworzy model `SVC` z dobranymi parametrami,
   * wykona walidację krzyżową (5-fold),
   * zwróci średnią dokładność walidacyjną jako wartość celu.
4. Utwórz study używając `TPESampler`.
5. Uruchom optymalizację na 40 próbach.
6. Wypisz najlepsze hiperparametry oraz najlepszy uzyskany wynik.
7. Wytrenuj finalny model na pełnym zbiorze treningowym z najlepszymi hiperparametrami i oceń go na zbiorze testowym — wypisz finalny accuracy.

Wyniki powinny pokazać:

* najlepsze hiperparametry wybrane przez Optunę,
* wynik walidacji,
* wynik na zbiorze testowym.

In [3]:
import optuna
from sklearn.svm import SVC
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score
import numpy as np

# 1) Wczytanie danych Wine
X, y = load_wine(return_X_y=True)

# 2) Podział danych (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# 3) Funkcja celu Optuny
def objective(trial):
    params = {
        "C": trial.suggest_float("C", 1e-3, 1e3, log=True),
        "gamma": trial.suggest_float("gamma", 1e-4, 10, log=True),
        "kernel": trial.suggest_categorical("kernel", ["rbf", "poly", "sigmoid"])
    }

    model = SVC(**params)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="accuracy")

    return scores.mean()

# 4) Utworzenie study
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))

# 5) Optymalizacja (40 prób)
study.optimize(objective, n_trials=40, show_progress_bar=True)

print("Najlepsze parametry:", study.best_params)
print("Najlepszy wynik CV:", study.best_value)

# 6) Trening finalnego modelu na najlepszych parametrach
final_model = SVC(**study.best_params)
final_model.fit(X_train, y_train)

# 7) Ocena na zbiorze testowym
preds = final_model.predict(X_test)
print("Accuracy na teście:", accuracy_score(y_test, preds))

/Users/bartek/Projects/courses-uwr/Warsztat AI/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-11-16 19:21:53,732] A new study created in memory with name: no-name-583e7132-9e29-4c29-a4e5-245aa912d9e8
Best trial: 0. Best value: 0.401232:  10%|█         | 4/40 [00:00<00:00, 62.28it/s]

[I 2026-11-16 19:21:53,757] Trial 0 finished with value: 0.4012315270935961 and parameters: {'C': 0.17670169402947947, 'gamma': 5.669849511478849, 'kernel': 'rbf'}. Best is trial 0 with value: 0.4012315270935961.
[I 2026-11-16 19:21:53,769] Trial 1 finished with value: 0.4012315270935961 and parameters: {'C': 0.00862913219007186, 'gamma': 0.00019517224641449495, 'kernel': 'rbf'}. Best is trial 0 with value: 0.4012315270935961.
[I 2026-11-16 19:21:53,780] Trial 2 finished with value: 0.4012315270935961 and parameters: {'C': 0.0013289448722869186, 'gamma': 7.0721141314722304, 'kernel': 'rbf'}. Best is trial 0 with value: 0.4012315270935961.
[I 2026-11-16 19:21:53,792] Trial 3 finished with value: 0.4012315270935961 and parameters: {'C': 0.012601639723276799, 'gamma': 0.003320559103751959, 'kernel': 'rbf'}. Best is trial 0 with value: 0.4012315270935961.
[I 2026-11-16 19:21:53,802] Trial 4 finished with value: 0.4012315270935961 and parameters: {'C': 4.68940096353769, 'gamma': 0.000498275

Best trial: 5. Best value: 0.943103:  15%|█▌        | 6/40 [00:00<00:02, 15.46it/s]

[I 2026-11-16 19:21:54,125] Trial 5 finished with value: 0.9431034482758621 and parameters: {'C': 51.41096648805749, 'gamma': 0.0009962513222055117, 'kernel': 'poly'}. Best is trial 5 with value: 0.9431034482758621.
[I 2026-11-16 19:21:54,139] Trial 6 finished with value: 0.4012315270935961 and parameters: {'C': 4.418441521199721, 'gamma': 0.0007122305833333872, 'kernel': 'sigmoid'}. Best is trial 5 with value: 0.9431034482758621.


Best trial: 5. Best value: 0.943103:  20%|██        | 8/40 [00:00<00:02, 10.73it/s]

[I 2026-11-16 19:21:54,431] Trial 7 finished with value: 0.9431034482758621 and parameters: {'C': 70.85721663941601, 'gamma': 0.0033347927286375843, 'kernel': 'poly'}. Best is trial 5 with value: 0.9431034482758621.


... (38 outputs left out)

Best trial: 14. Best value: 0.950246:  90%|█████████ | 36/40 [00:06<00:00,  6.11it/s]

[I 2026-11-16 19:22:00,197] Trial 34 finished with value: 0.9431034482758621 and parameters: {'C': 2.4243822669635553, 'gamma': 0.00014485491793096098, 'kernel': 'poly'}. Best is trial 14 with value: 0.9502463054187192.
[I 2026-11-16 19:22:00,324] Trial 35 finished with value: 0.9366995073891626 and parameters: {'C': 0.00131676096410931, 'gamma': 0.00036669867465314053, 'kernel': 'poly'}. Best is trial 14 with value: 0.9502463054187192.


Best trial: 14. Best value: 0.950246: 100%|██████████| 40/40 [00:06<00:00,  5.80it/s]

[I 2026-11-16 19:22:00,601] Trial 36 finished with value: 0.9431034482758621 and parameters: {'C': 0.11190743315319221, 'gamma': 0.0014828439029926262, 'kernel': 'poly'}. Best is trial 14 with value: 0.9502463054187192.
[I 2026-11-16 19:22:00,611] Trial 37 finished with value: 0.4012315270935961 and parameters: {'C': 0.004429786628753929, 'gamma': 0.00024028897042420065, 'kernel': 'sigmoid'}. Best is trial 14 with value: 0.9502463054187192.
[I 2026-11-16 19:22:00,622] Trial 38 finished with value: 0.4012315270935961 and parameters: {'C': 0.0019457065641773949, 'gamma': 0.0406329125454954, 'kernel': 'sigmoid'}. Best is trial 14 with value: 0.9502463054187192.
[I 2026-11-16 19:22:00,633] Trial 39 finished with value: 0.4012315270935961 and parameters: {'C': 1.0025605215456377, 'gamma': 0.0002248830433555255, 'kernel': 'sigmoid'}. Best is trial 14 with value: 0.9502463054187192.
Najlepsze parametry: {'C': 0.30208901223476553, 'gamma': 0.00015164979202784659, 'kernel': 'poly'}
Najlepszy wy